Model Development and Comparison — Group 2026-DS-77

Responsible member: IT24102789

Preprocessing support: IT24103676

Protocol review: IT24103677

Objective: Compare a sensible baseline and at least three alternative supervised classification models for recorded customer churn.

Investigate whether increased model complexity provides useful predictive improvement relative to interpretability.

We will reuse the existing development subset and Member 3’s shared preprocessing module.

 All learned preprocessing will be fitted inside the training portion of each cross-validation fold.

Mean validation average precision will guide model selection.

The selected configuration’s out-of-fold development scores will support threshold selection by maximising F1.

The reserved test set will remain separate until the procedure is frozen.

In [1]:
from pathlib import Path

drive_root = Path("/content/drive")

for filename in [
    "development.csv",
    "telco_preprocessing.py",
    "preprocessing_run_record.json"
]:
    matches = list(drive_root.rglob(filename))

    print(f"\n{filename}:")
    if matches:
        for m in matches:
            print(m)
    else:
        print("NOT FOUND")


development.csv:
NOT FOUND

telco_preprocessing.py:
NOT FOUND

preprocessing_run_record.json:
NOT FOUND


In [2]:
#begin Member 4’s work T24102789, using Member 3’s shared preprocessing module. IT24103676 supports preprocessing integration, and IT24103677 reviews the evaluation procedure
#Set up the modelling notebook and verify the handover.
#first step is setup and verification. We are not training models yet.

#Mount Drive and locate the handover files

from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/ML 2026-DS-77 Telco Customer Churn"
)

DEV_PATH = PROJECT_DIR / "data" / "splits" / "development.csv"
SRC_DIR = PROJECT_DIR / "src"
MODULE_PATH = SRC_DIR / "telco_preprocessing.py"

PREPROCESSING_RECORD_PATH = (
    PROJECT_DIR
    / "outputs"
    / "tables"
    / "preprocessing_run_record.json"
)

# Check the files required for the modelling handover.
# We do not load the reserved test file.
for path in [
    DEV_PATH,
    MODULE_PATH,
    PREPROCESSING_RECORD_PATH,
]:
    if not path.is_file():
        raise FileNotFoundError(f"Required handover file missing: {path}")

print("All required handover files found.")

Mounted at /content/drive
All required handover files found.


In [3]:
#Verify that the development data and module match Member 3’s record
import hashlib
import json

def file_sha256(path):
    """Return a fingerprint of the file's current contents."""
    return hashlib.sha256(path.read_bytes()).hexdigest()

preprocessing_record = json.loads(
    PREPROCESSING_RECORD_PATH.read_text(encoding="utf-8")
)

# Confirm that we received the same files used in Member 3's checks.
assert file_sha256(DEV_PATH) == preprocessing_record["development_sha256"], (
    "The development file differs from Member 3's recorded version. "
    "Review the change before continuing."
)

assert file_sha256(MODULE_PATH) == preprocessing_record["module_sha256"], (
    "The preprocessing module differs from Member 3's recorded version. "
    "Review the change before continuing."
)

print("Development file matches Member 3's record.")
print("Preprocessing module matches Member 3's record.")

#Why this matters: matching names alone does not guarantee matching contents. These checks help prevent accidental changes between members. A matching fingerprint establishes file identity, not scientific correctness.

Development file matches Member 3's record.
Preprocessing module matches Member 3's record.


In [4]:
#Import the shared module and load the development data

import importlib
import platform
import sys

import numpy as np
import pandas as pd
import sklearn

# Allow this notebook to import code from the shared src folder.
src_location = str(SRC_DIR)
if src_location not in sys.path:
    sys.path.insert(0, src_location)

importlib.invalidate_caches()

import telco_preprocessing as telco_prep
telco_prep = importlib.reload(telco_prep)

# Make sure we imported this project's module.
assert Path(telco_prep.__file__).resolve() == MODULE_PATH.resolve()

# Read only the existing development data.
dev = pd.read_csv(
    DEV_PATH,
    dtype={"customerID": "string"}
)

assert dev.shape == (5634, 21)
assert dev.columns.is_unique
assert dev["customerID"].notna().all()
assert dev["customerID"].is_unique
assert dev["Churn"].notna().all()
assert dev["Churn"].value_counts().to_dict() == {
    "No": 4139,
    "Yes": 1495,
}

# Preserve identifiers for matching predictions to customers later.
customer_ids = dev["customerID"].copy()

# Positive class = recorded churn.
y = dev["Churn"].map({"No": 0, "Yes": 1}).astype("int64")

# Keep the original predictor values.
# The shared pipeline will parse TotalCharges internally.
X = dev.drop(columns=["customerID", "Churn"]).copy()

assert X.shape == (5634, 19)
assert set(X.columns) == set(telco_prep.PREDICTOR_COLUMNS)
assert X.index.equals(y.index)
assert X.index.equals(customer_ids.index)

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)

print("\nDevelopment records:", len(X))
print("Original predictors:", X.shape[1])
print("Shared preprocessing module imported successfully.")

display(
    y.value_counts()
    .sort_index()
    .rename_axis("Churn: 0 = No, 1 = Yes")
    .to_frame("Customers")
)

# Flag environment differences for review without changing packages.
recorded_sklearn = preprocessing_record.get("scikit_learn_version")
if sklearn.__version__ != recorded_sklearn:
    print(
        "\nVersion difference to review:",
        f"Member 3 used scikit-learn {recorded_sklearn};",
        f"this session uses {sklearn.__version__}."
    )

#Important: Member 4’s X should have 19 original columns, not 46 already-transformed columns. Each model pipeline will create its encoded features during fitting.

Python: 3.13.16
pandas: 2.2.3
NumPy: 2.1.3
scikit-learn: 1.6.1

Development records: 5634
Original predictors: 19
Shared preprocessing module imported successfully.


,Customers
"Churn: 0 = No, 1 = Yes",
0,4139
1,1495


The handover is verified: Member 4 has the correct development data, the correct preprocessing module and the same scikit-learn version used by Member 3.

Next, choose and define the model candidates.

**Initial model candidates and justification**

We select a class-prior dummy classifier as the baseline and three alternative methods: regularised logistic regression, a shallow decision tree and a random forest.

The dummy classifier provides a reference that does not distinguish customers using their attributes. Logistic regression provides a relatively simple additive model. A shallow decision tree provides inspectable branching rules and can capture interactions. Random forest investigates whether combining many trees provides useful predictive improvement despite greater complexity.

These methods support our investigation into predictive performance and interpretability. We do not assume that the most complex method will perform best.

Logistic regression will use numerical scaling. The decision tree and random forest will use the unscaled numerical preparation route. All methods will use fresh instances of the shared preprocessing module inside their complete pipelines.

Initial configurations are starting points, not optimised settings. Model comparison, bounded tuning and predictive evaluation remain pending. No class weighting or resampling is applied in the initial comparison.

Understand the baseline:

Suppose a training fold contains approximately 26.5% churners.
The dummy baseline gives every validation customer approximately the same score:

Customer A → 0.265

Customer B → 0.265

Customer C → 0.265

It learns the training class proportion but does not use customer characteristics to rank customers.

This is useful because your real models should demonstrate that those characteristics provide useful information.

In [5]:
#Define the four complete pipelines

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

# A reproducibility setting, not a seed selected for better results.
MODEL_SEED = 42


def make_candidate_pipeline(estimator, scale_numeric):
    """Combine fresh preprocessing with an unfitted classifier."""
    return Pipeline([
        (
            "preprocessing",
            telco_prep.make_model_preprocessor(
                scale_numeric=scale_numeric
            ),
        ),
        ("model", estimator),
    ])


candidate_pipelines = {
    # Baseline: every customer receives the training class-prior score.
    # Preprocessing is included for a consistent pipeline interface,
    # although this classifier ignores the predictor values.
    "Dummy prior": make_candidate_pipeline(
        DummyClassifier(strategy="prior"),
        scale_numeric=False,
    ),

    # L2 regularisation discourages excessively large coefficients.
    # Numerical scaling makes the regularisation treatment more comparable
    # across numerical features with different units.
    "Logistic regression": make_candidate_pipeline(
        LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="lbfgs",
            max_iter=2000,
            class_weight=None,
        ),
        scale_numeric=True,
    ),

    # Restrict the initial tree to keep the rules relatively compact.
    "Shallow decision tree": make_candidate_pipeline(
        DecisionTreeClassifier(
            max_depth=4,
            min_samples_leaf=20,
            class_weight=None,
            random_state=MODEL_SEED,
        ),
        scale_numeric=False,
    ),

    # An ensemble candidate with a fixed initial number of trees.
    # n_jobs=1 avoids nested parallel processing if CV is parallelised later.
    "Random forest": make_candidate_pipeline(
        RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            min_samples_leaf=5,
            max_features="sqrt",
            class_weight=None,
            random_state=MODEL_SEED,
            n_jobs=1,
        ),
        scale_numeric=False,
    ),
}

# Display the definitions without fitting any model.
candidate_summary = pd.DataFrame([
    {
        "Candidate": name,
        "Classifier": pipeline.named_steps["model"].__class__.__name__,
        "Numerical scaling": name == "Logistic regression",
    }
    for name, pipeline in candidate_pipelines.items()
])

display(candidate_summary)

print("Four complete candidate pipelines defined.")
print("No models have been fitted.")

,Candidate,Classifier,Numerical scaling
0,Dummy prior,DummyClassifier,False
1,Logistic regression,LogisticRegression,True
2,Shallow decision tree,DecisionTreeClassifier,False
3,Random forest,RandomForestClassifier,False


Four complete candidate pipelines defined.
No models have been fitted.


What the initial settings mean
- Logistic regression C=1.0: a starting regularisation setting. We can compare other strengths during bounded tuning.
- Tree max_depth=4: limits the number of branching levels, making the initial tree easier to inspect.
- Tree min_samples_leaf=20: prevents a final branch from describing fewer than 20 training records.
- Forest n_estimators=200: starts with 200 trees—a manageable computational choice, not a proven optimum.
- Forest min_samples_leaf=5: prevents leaves containing only one or a few training examples.
- Forest max_depth=None: does not impose a fixed depth limit; its leaf-size restriction still constrains growth.

These numbers are explicit starting assumptions. We cannot describe them as the best settings before comparison and tuning.

**Shared development cross-validation**

We use five stratified development cross-validation folds with shuffling and random seed 42. All candidate models use the same fold membership.

Each development customer appears in validation exactly once and in training four times. Within each fold, the complete pipeline learns preprocessing and model parameters using only the training records.

We record validation average precision for each fold and calculate its arithmetic mean for model comparison. Fold variation will be reported descriptively; it will not be treated as proof of statistical significance.

This cross-validation supports development decisions. It does not replace the reserved test assessment or establish future-period performance.


In [6]:
#we’ll create and save the five cross-validation folds.
#Create and check the folds

from sklearn.model_selection import StratifiedKFold

CV_SEED = 42
N_SPLITS = 5

cv_splitter = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=CV_SEED,
)

# Materialise the splits once.
# Every candidate will receive this same list of row positions.
cv_splits = list(cv_splitter.split(X, y))

# Record which validation fold each customer belongs to.
validation_fold = np.full(len(X), -1, dtype=int)
validation_appearances = np.zeros(len(X), dtype=int)

fold_summary_rows = []

for fold_number, (train_positions, valid_positions) in enumerate(
    cv_splits,
    start=1,
):
    # Training and validation must not overlap within this fold.
    assert np.intersect1d(train_positions, valid_positions).size == 0
    assert len(train_positions) + len(valid_positions) == len(X)

    validation_fold[valid_positions] = fold_number
    validation_appearances[valid_positions] += 1

    y_train_fold = y.iloc[train_positions]
    y_valid_fold = y.iloc[valid_positions]

    # Both classes must be represented.
    assert set(y_train_fold.unique()) == {0, 1}
    assert set(y_valid_fold.unique()) == {0, 1}

    fold_summary_rows.append({
        "Fold": fold_number,
        "Training records": len(train_positions),
        "Validation records": len(valid_positions),
        "Validation churners": int(y_valid_fold.sum()),
        "Validation non-churners": int((y_valid_fold == 0).sum()),
        "Validation churn percentage": round(
            100 * y_valid_fold.mean(), 2
        ),
    })

# Every development customer must be validated exactly once.
assert np.all(validation_appearances == 1)
assert np.all(validation_fold >= 1)

fold_summary = pd.DataFrame(fold_summary_rows)

# Customer IDs let us trace the allocation without using IDs as predictors.
cv_manifest = pd.DataFrame({
    "customerID": customer_ids.to_numpy(),
    "validation_fold": validation_fold,
})

assert cv_manifest["customerID"].is_unique
assert len(cv_manifest) == len(X)

display(fold_summary)

print("All five folds checked.")
print("Every development customer appears in validation exactly once.")
print("No models have been fitted.")

#We will use stratification to keep the churn proportion approximately similar across groups, with shuffling and seed 42 for reproducibility.
#The seed is a repeatability choice, not a setting selected for better scores.

,Fold,Training records,Validation records,Validation churners,Validation non-churners,Validation churn percentage
0,1,4507,1127,299,828,26.53
1,2,4507,1127,299,828,26.53
2,3,4507,1127,299,828,26.53
3,4,4507,1127,299,828,26.53
4,5,4508,1126,299,827,26.55


All five folds checked.
Every development customer appears in validation exactly once.
No models have been fitted.


In [7]:

#Save the fold membership and settings
CV_MANIFEST_PATH = (
    PROJECT_DIR / "data" / "splits" / "development_cv_manifest.csv"
)
CV_SETTINGS_PATH = (
    PROJECT_DIR / "data" / "splits" / "development_cv_settings.json"
)

cv_settings = {
    "n_splits": N_SPLITS,
    "shuffle": True,
    "random_state": CV_SEED,
    "development_sha256": file_sha256(DEV_PATH),
    "positive_class": 1,
    "positive_class_meaning": "Churn = Yes",
}

# Check existing files BEFORE writing anything.
# Stop if an earlier allocation or its settings differ.
if CV_MANIFEST_PATH.exists():
    saved_manifest = pd.read_csv(
        CV_MANIFEST_PATH,
        dtype={"customerID": "string"},
    )

    pd.testing.assert_frame_equal(
        saved_manifest,
        cv_manifest.astype({"customerID": "string"}),
    )

if CV_SETTINGS_PATH.exists():
    saved_settings = json.loads(
        CV_SETTINGS_PATH.read_text(encoding="utf-8")
    )
    assert saved_settings == cv_settings, (
        "Existing CV settings differ. Review before continuing."
    )

# Create missing files; preserve identical existing ones.
if not CV_MANIFEST_PATH.exists():
    cv_manifest.to_csv(CV_MANIFEST_PATH, index=False)

if not CV_SETTINGS_PATH.exists():
    CV_SETTINGS_PATH.write_text(
        json.dumps(cv_settings, indent=2),
        encoding="utf-8",
    )

print("CV manifest saved or verified:", CV_MANIFEST_PATH.name)
print("CV settings saved or verified:", CV_SETTINGS_PATH.name)

#These are additional records inside data/splits/. They do not replace development.csv, test.csv or the original split manifest.

CV manifest saved or verified: development_cv_manifest.csv
CV settings saved or verified: development_cv_settings.json


**Bounded tuning and selection rules**

Before inspecting model-comparison results, we specify bounded searches for logistic regression, decision tree and random forest. The searches vary regularisation or tree-complexity controls and compare no class weighting with balanced class weighting. No resampling is introduced.

All three alternative model families will be tuned using the same five saved development folds and mean validation average precision. Other settings remain fixed as recorded in the initial candidate definitions. The dummy baseline remains untuned.

Selection uses unrounded mean validation average precision. Exact ties within a model are resolved using the first configuration in the saved ordered list. Exact ties between model families follow this order: dummy baseline, logistic regression, decision tree, random forest.

For the later threshold-selection stage, we will maximise F1 on out-of-fold development scores. Exact ties in maximum F1 will be resolved by choosing the highest threshold. Customers will be flagged when their score meets or exceeds that threshold.

The search is limited and does not establish a globally optimal model. Development results support selection and are not independent final performance estimates. Any necessary protocol revision will be documented with its reason.

Before training, we’ll record which settings we will try and how we will resolve ties.

This prevents us from changing the rules simply because we prefer a particular result.

Set a manageable tuning plan.

A hyperparameter is a setting we choose before a model learns—for example, how deep a tree may grow.

In [8]:
#Define and save the plan

from itertools import product

# Each dictionary describes one complete configuration to try.
# List order also supplies the declared within-model exact-tie rule.
# "model__" refers to the classifier step inside our complete pipeline.
tuning_configs = {
    "Logistic regression": [
        {
            "model__C": c,
            "model__class_weight": weight,
        }
        for c, weight in product(
            [0.1, 1.0, 10.0],
            [None, "balanced"],
        )
    ],

    "Shallow decision tree": [
        {
            "model__max_depth": depth,
            "model__min_samples_leaf": leaf_size,
            "model__class_weight": weight,
        }
        for depth, leaf_size, weight in product(
            [3, 4, 6],
            [20, 10],
            [None, "balanced"],
        )
    ],

    "Random forest": [
        {
            "model__max_depth": depth,
            "model__min_samples_leaf": leaf_size,
            "model__class_weight": weight,
        }
        for depth, leaf_size, weight in product(
            [10, None],
            [10, 5],
            [None, "balanced"],
        )
    ],
}

MODEL_TIE_ORDER = [
    "Dummy prior",
    "Logistic regression",
    "Shallow decision tree",
    "Random forest",
]

# Check that the requested parameter names exist in each pipeline.
# This validates the plan without training anything.
for name, configurations in tuning_configs.items():
    valid_parameters = candidate_pipelines[name].get_params(deep=True)

    for configuration in configurations:
        assert set(configuration).issubset(valid_parameters), (
            f"Unknown parameter in {name}: {configuration}"
        )

search_summary = pd.DataFrame([
    {
        "Model": name,
        "Configurations": len(configurations),
        "Folds per configuration": N_SPLITS,
        "CV training fits": len(configurations) * N_SPLITS,
    }
    for name, configurations in tuning_configs.items()
])

# Save settings, data/code identities and selection rules together.
model_protocol = {
    "development_sha256": file_sha256(DEV_PATH),
    "preprocessing_module_sha256": file_sha256(MODULE_PATH),
    "cv_manifest_sha256": file_sha256(CV_MANIFEST_PATH),
    "cv_settings": cv_settings,
    "scikit_learn_version": sklearn.__version__,
    "positive_class": 1,
    "selection_metric": "mean_validation_average_precision",
    "use_unrounded_scores_for_selection": True,
    "initial_estimator_settings": {
        name: pipeline.named_steps["model"].get_params(deep=False)
        for name, pipeline in candidate_pipelines.items()
    },
    "numerical_scaling": {
        name: name == "Logistic regression"
        for name in candidate_pipelines
    },
    "ordered_tuning_configurations": tuning_configs,
    "within_model_exact_tie": "first configuration in saved list",
    "between_model_exact_tie_order": MODEL_TIE_ORDER,
    "threshold_rule": "maximise F1 on selected configuration OOF scores",
    "threshold_exact_tie": "highest threshold among exact F1 maximisers",
    "flag_rule": "score >= threshold",
    "resampling": False,
}

PROTOCOL_DIR = PROJECT_DIR / "outputs" / "tables"
PROTOCOL_DIR.mkdir(parents=True, exist_ok=True)
PROTOCOL_PATH = PROTOCOL_DIR / "model_development_protocol.json"

# Preserve an existing plan instead of silently replacing it.
if PROTOCOL_PATH.exists():
    saved_protocol = json.loads(
        PROTOCOL_PATH.read_text(encoding="utf-8")
    )
    assert saved_protocol == model_protocol, (
        "The saved protocol differs. Review and document any revision "
        "before replacing it."
    )
    print("Reusing the identical modelling protocol.")
else:
    PROTOCOL_PATH.write_text(
        json.dumps(model_protocol, indent=2),
        encoding="utf-8",
    )
    print("Saved modelling protocol:", PROTOCOL_PATH.name)

display(search_summary)

print("Total tuning configurations:", sum(map(len, tuning_configs.values())))
print("Total tuning CV fits:", int(search_summary["CV training fits"].sum()))
print("No models have been fitted.")

Reusing the identical modelling protocol.


,Model,Configurations,Folds per configuration,CV training fits
0,Logistic regression,6,5,30
1,Shallow decision tree,12,5,60
2,Random forest,8,5,40


Total tuning configurations: 26
Total tuning CV fits: 130
No models have been fitted.


**Initial model comparison**

We evaluate the four initial candidate pipelines using the five saved development cross-validation folds.

For every fold, a fresh copy of the complete pipeline is fitted on the training records. Validation records receive only the transformations learned from those training records.

We record training and validation average precision, together with pipeline fitting time. Mean validation average precision is the primary comparison measure. Training scores help identify possible overfitting, while fitting time provides a descriptive computational comparison.

These are initial development results. They do not select the final model before the planned tuning comparisons, and they do not replace the reserved test assessment.

In [9]:
#run the first actual model comparison.
#This stage uses the four initial configurations, not the 26 tuning configurations yet. It will perform 20 fits: four candidates × five folds.
#Run the initial comparison

from time import perf_counter
import warnings

from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import average_precision_score

initial_fold_rows = []

for model_name, pipeline_definition in candidate_pipelines.items():
    print(f"\nEvaluating: {model_name}")

    for fold_number, (train_positions, valid_positions) in enumerate(
        cv_splits,
        start=1,
    ):
        X_train_fold = X.iloc[train_positions]
        X_valid_fold = X.iloc[valid_positions]

        y_train_fold = y.iloc[train_positions]
        y_valid_fold = y.iloc[valid_positions]

        # A fresh, unfitted copy for this fold.
        # This includes fresh preprocessing and a fresh classifier.
        fitted_pipeline = clone(pipeline_definition)

        start_time = perf_counter()

        # Stop if optimisation fails to converge instead of silently
        # treating an unfinished fit as a satisfactory result.
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            fitted_pipeline.fit(X_train_fold, y_train_fold)

        fit_seconds = perf_counter() - start_time

        # Find the probability column corresponding to Churn = Yes.
        # Do not assume the column position without checking.
        classes = fitted_pipeline.named_steps["model"].classes_
        positive_columns = np.flatnonzero(classes == 1)
        assert len(positive_columns) == 1
        positive_column = int(positive_columns[0])

        # Use continuous scores for AP, not thresholded Yes/No predictions.
        train_scores = fitted_pipeline.predict_proba(
            X_train_fold
        )[:, positive_column]

        valid_scores = fitted_pipeline.predict_proba(
            X_valid_fold
        )[:, positive_column]

        assert np.isfinite(train_scores).all()
        assert np.isfinite(valid_scores).all()

        train_ap = average_precision_score(
            y_train_fold, train_scores
        )
        valid_ap = average_precision_score(
            y_valid_fold, valid_scores
        )

        # A constant-score baseline should have AP equal to
        # the validation subset's positive-class proportion.
        if model_name == "Dummy prior":
            np.testing.assert_allclose(
                valid_ap,
                y_valid_fold.mean(),
                atol=1e-12,
            )

        initial_fold_rows.append({
            "Model": model_name,
            "Fold": fold_number,
            "Training records": len(train_positions),
            "Validation records": len(valid_positions),
            "Training AP": train_ap,
            "Validation AP": valid_ap,
            "Fit seconds": fit_seconds,
        })

        print(
            f"  Fold {fold_number}: "
            f"training AP = {train_ap:.4f}, "
            f"validation AP = {valid_ap:.4f}"
        )

# Confirm that all four models completed all five folds.
initial_fold_results = pd.DataFrame(initial_fold_rows)

assert len(initial_fold_results) == len(candidate_pipelines) * N_SPLITS
assert (
    initial_fold_results.groupby("Model")["Fold"].nunique() == N_SPLITS
).all()

# The last fold's fitted object is not our final selected model.
del fitted_pipeline

print("\nInitial comparison completed: 20 successful fits.")

#clone() creates an unfitted copy, so one fold does not reuse another fold’s learned preparation or model
#Average precision is calculated from the continuous churn scores. An AP of 0.60 does not mean 60% accuracy or 60% recall. It summarises precision–recall performance across score thresholds


Evaluating: Dummy prior
  Fold 1: training AP = 0.2654, validation AP = 0.2653
  Fold 2: training AP = 0.2654, validation AP = 0.2653
  Fold 3: training AP = 0.2654, validation AP = 0.2653
  Fold 4: training AP = 0.2654, validation AP = 0.2653
  Fold 5: training AP = 0.2653, validation AP = 0.2655

Evaluating: Logistic regression
  Fold 1: training AP = 0.6695, validation AP = 0.6551
  Fold 2: training AP = 0.6598, validation AP = 0.6922
  Fold 3: training AP = 0.6676, validation AP = 0.6652
  Fold 4: training AP = 0.6719, validation AP = 0.6470
  Fold 5: training AP = 0.6687, validation AP = 0.6441

Evaluating: Shallow decision tree
  Fold 1: training AP = 0.6142, validation AP = 0.5736
  Fold 2: training AP = 0.6134, validation AP = 0.6213
  Fold 3: training AP = 0.6056, validation AP = 0.6253
  Fold 4: training AP = 0.6242, validation AP = 0.5762
  Fold 5: training AP = 0.6155, validation AP = 0.6127

Evaluating: Random forest
  Fold 1: training AP = 0.8341, validation AP = 0.6598


In [10]:
#Summarise and save the results
initial_summary = (
    initial_fold_results
    .groupby("Model", sort=False)
    .agg(
        Mean_training_AP=("Training AP", "mean"),
        Mean_validation_AP=("Validation AP", "mean"),
        SD_validation_AP=("Validation AP", "std"),
        Min_validation_AP=("Validation AP", "min"),
        Max_validation_AP=("Validation AP", "max"),
        Mean_fit_seconds=("Fit seconds", "mean"),
    )
    .reset_index()
)

# A descriptive gap, not a separate model-selection metric.
initial_summary["Training_validation_AP_gap"] = (
    initial_summary["Mean_training_AP"]
    - initial_summary["Mean_validation_AP"]
)

# Sort using unrounded scores and the previously declared exact-tie order.
tie_priority = {
    name: position
    for position, name in enumerate(MODEL_TIE_ORDER)
}

initial_summary["_tie_priority"] = initial_summary["Model"].map(
    tie_priority
)

initial_summary = (
    initial_summary
    .sort_values(
        ["Mean_validation_AP", "_tie_priority"],
        ascending=[False, True],
    )
    .drop(columns="_tie_priority")
    .reset_index(drop=True)
)

# Save full-precision results.
# Rounding is used only for the notebook display.
RESULTS_DIR = PROJECT_DIR / "outputs" / "tables"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

initial_fold_results.to_csv(
    RESULTS_DIR / "initial_model_cv_fold_results.csv",
    index=False,
)

initial_summary.to_csv(
    RESULTS_DIR / "initial_model_cv_summary.csv",
    index=False,
)

print("Initial model comparison:")
display(initial_summary.round(4))

print("Validation AP for each fold:")
display(
    initial_fold_results.pivot(
        index="Model",
        columns="Fold",
        values="Validation AP",
    ).round(4)
)

print("Saved initial fold results and summary.")
print("These are development results; final model selection is pending.")

Initial model comparison:


,Model,Mean_training_AP,Mean_validation_AP,SD_validation_AP,Min_validation_AP,Max_validation_AP,Mean_fit_seconds,Training_validation_AP_gap
0,Logistic regression,0.6675,0.6607,0.0194,0.6441,0.6922,0.1999,0.0068
1,Random forest,0.8343,0.6592,0.0268,0.6231,0.6867,1.2048,0.1751
2,Shallow decision tree,0.6146,0.6018,0.0250,0.5736,0.6253,0.0921,0.0128
3,Dummy prior,0.2654,0.2654,0.0001,0.2653,0.2655,0.1370,-0.0000


Validation AP for each fold:


Fold,1,2,3,4,5
Model,,,,,
Dummy prior,0.2653,0.2653,0.2653,0.2653,0.2655
Logistic regression,0.6551,0.6922,0.6652,0.6470,0.6441
Random forest,0.6598,0.6867,0.6829,0.6231,0.6433
Shallow decision tree,0.5736,0.6213,0.6253,0.5762,0.6127


Saved initial fold results and summary.
These are development results; final model selection is pending.


**Interpretation of the initial comparison**

All three supervised alternatives exceeded the class-prior baseline in mean validation average precision. Logistic regression achieved 0.6607, random forest 0.6592 and the shallow decision tree 0.6018, compared with 0.2654 for the dummy baseline.

Logistic regression and random forest performed similarly. Their mean difference was approximately 0.0015 AP, with logistic regression scoring higher in three folds and random forest in two. These results do not establish statistical significance or equivalence.

Random forest showed a larger training–validation AP gap of approximately 0.1751, suggesting greater overfitting in its initial configuration. Logistic regression’s gap was approximately 0.0068.

The initial results do not demonstrate a validation advantage from the random forest’s additional complexity. We will proceed with the previously specified tuning search for all three alternative model families before selecting the preferred configuration. The reserved test set remains unused.

In [11]:
#Run the planned tuning search
# Check that the tuning plan in memory still matches the saved protocol.
saved_protocol = json.loads(
    PROTOCOL_PATH.read_text(encoding="utf-8")
)

assert (
    saved_protocol["ordered_tuning_configurations"]
    == tuning_configs
), "Tuning configurations differ from the saved protocol."

assert (
    saved_protocol["development_sha256"] == file_sha256(DEV_PATH)
), "Development file changed."

assert (
    saved_protocol["preprocessing_module_sha256"]
    == file_sha256(MODULE_PATH)
), "Preprocessing module changed."

assert (
    saved_protocol["cv_manifest_sha256"]
    == file_sha256(CV_MANIFEST_PATH)
), "Saved CV membership changed."

tuning_fold_rows = []

for model_name, configurations in tuning_configs.items():
    print(f"\nTuning: {model_name}")

    for config_id, configuration in enumerate(configurations, start=1):
        configuration_fold_aps = []

        for fold_number, (train_positions, valid_positions) in enumerate(
            cv_splits,
            start=1,
        ):
            # New preprocessing and classifier for every fit.
            fitted_pipeline = clone(
                candidate_pipelines[model_name]
            ).set_params(**configuration)

            X_train_fold = X.iloc[train_positions]
            X_valid_fold = X.iloc[valid_positions]
            y_train_fold = y.iloc[train_positions]
            y_valid_fold = y.iloc[valid_positions]

            start_time = perf_counter()

            with warnings.catch_warnings():
                warnings.simplefilter("error", ConvergenceWarning)
                fitted_pipeline.fit(X_train_fold, y_train_fold)

            fit_seconds = perf_counter() - start_time

            classes = fitted_pipeline.named_steps["model"].classes_
            positive_columns = np.flatnonzero(classes == 1)
            assert len(positive_columns) == 1
            positive_column = int(positive_columns[0])

            train_scores = fitted_pipeline.predict_proba(
                X_train_fold
            )[:, positive_column]

            valid_scores = fitted_pipeline.predict_proba(
                X_valid_fold
            )[:, positive_column]

            assert np.isfinite(train_scores).all()
            assert np.isfinite(valid_scores).all()

            train_ap = average_precision_score(
                y_train_fold, train_scores
            )
            valid_ap = average_precision_score(
                y_valid_fold, valid_scores
            )

            tuning_fold_rows.append({
                "Model": model_name,
                "Configuration": config_id,
                "Parameters": json.dumps(configuration, sort_keys=True),
                "Fold": fold_number,
                "Training AP": train_ap,
                "Validation AP": valid_ap,
                "Fit seconds": fit_seconds,
            })

            configuration_fold_aps.append(valid_ap)

        print(
            f"  Configuration {config_id}/{len(configurations)}: "
            f"mean validation AP = "
            f"{np.mean(configuration_fold_aps):.4f}"
        )

tuning_fold_results = pd.DataFrame(tuning_fold_rows)

expected_fits = (
    sum(len(configs) for configs in tuning_configs.values()) * N_SPLITS
)
assert len(tuning_fold_results) == expected_fits

# Every configuration must have results for all five folds.
assert (
    tuning_fold_results
    .groupby(["Model", "Configuration"])["Fold"]
    .nunique()
    .eq(N_SPLITS)
    .all()
)

# This last fitted fold model is not the final selected pipeline.
del fitted_pipeline

print(f"\nTuning completed: {expected_fits} successful fits.")


Tuning: Logistic regression
  Configuration 1/6: mean validation AP = 0.6596
  Configuration 2/6: mean validation AP = 0.6584
  Configuration 3/6: mean validation AP = 0.6607
  Configuration 4/6: mean validation AP = 0.6594
  Configuration 5/6: mean validation AP = 0.6601
  Configuration 6/6: mean validation AP = 0.6585

Tuning: Shallow decision tree
  Configuration 1/12: mean validation AP = 0.5639
  Configuration 2/12: mean validation AP = 0.5538
  Configuration 3/12: mean validation AP = 0.5639
  Configuration 4/12: mean validation AP = 0.5538
  Configuration 5/12: mean validation AP = 0.6018
  Configuration 6/12: mean validation AP = 0.5941
  Configuration 7/12: mean validation AP = 0.6019
  Configuration 8/12: mean validation AP = 0.5941
  Configuration 9/12: mean validation AP = 0.6124
  Configuration 10/12: mean validation AP = 0.6087
  Configuration 11/12: mean validation AP = 0.6071
  Configuration 12/12: mean validation AP = 0.6060

Tuning: Random forest
  Configuration 1/8:

In [12]:
#Summarise the tuning results
tuning_summary = (
    tuning_fold_results
    .groupby(["Model", "Configuration", "Parameters"], sort=False)
    .agg(
        Mean_training_AP=("Training AP", "mean"),
        Mean_validation_AP=("Validation AP", "mean"),
        SD_validation_AP=("Validation AP", "std"),
        Mean_fit_seconds=("Fit seconds", "mean"),
    )
    .reset_index()
)

tuning_summary["Training_validation_AP_gap"] = (
    tuning_summary["Mean_training_AP"]
    - tuning_summary["Mean_validation_AP"]
)

# Within each model:
# highest unrounded mean AP wins;
# an exact tie uses the earlier saved configuration.
best_tuned_per_model = (
    tuning_summary
    .sort_values(
        ["Model", "Mean_validation_AP", "Configuration"],
        ascending=[True, False, True],
    )
    .groupby("Model", sort=False)
    .head(1)
    .reset_index(drop=True)
)

# Save complete results, not just the winners.
tuning_fold_results.to_csv(
    RESULTS_DIR / "tuned_model_cv_fold_results.csv",
    index=False,
)
tuning_summary.to_csv(
    RESULTS_DIR / "tuned_model_cv_all_configurations.csv",
    index=False,
)
best_tuned_per_model.to_csv(
    RESULTS_DIR / "tuned_model_cv_best_per_model.csv",
    index=False,
)

print("Best searched configuration within each model family:")
display(best_tuned_per_model.round(4))

# Print parameter dictionaries separately so they are easy to read.
for _, row in best_tuned_per_model.iterrows():
    print(f"\n{row['Model']}")
    print("Configuration:", row["Configuration"])
    print("Parameters:", row["Parameters"])

print("\nSaved all tuning results and the best-per-model summary.")

Best searched configuration within each model family:


,Model,Configuration,Parameters,Mean_training_AP,Mean_validation_AP,SD_validation_AP,Mean_fit_seconds,Training_validation_AP_gap
0,Logistic regression,3,"{""model__C"": 1.0, ""model__class_weight"": null}",0.6675,0.6607,0.0194,0.1385,0.0068
1,Random forest,1,"{""model__class_weight"": null, ""model__max_dept...",0.7568,0.6662,0.0244,1.1985,0.0905
2,Shallow decision tree,9,"{""model__class_weight"": null, ""model__max_dept...",0.6836,0.6124,0.0213,0.0967,0.0712



Logistic regression
Configuration: 3
Parameters: {"model__C": 1.0, "model__class_weight": null}

Random forest
Configuration: 1
Parameters: {"model__class_weight": null, "model__max_depth": 10, "model__min_samples_leaf": 10}

Shallow decision tree
Configuration: 9
Parameters: {"model__class_weight": null, "model__max_depth": 6, "model__min_samples_leaf": 20}

Saved all tuning results and the best-per-model summary.


In [13]:
#Select the configuration using the unrounded results
# Build a comparison of the best searched configuration in each family.
selection_rows = []

for _, row in best_tuned_per_model.iterrows():
    selection_rows.append({
        "Model": row["Model"],
        "Configuration": int(row["Configuration"]),
        "Parameters": row["Parameters"],
        "Mean_validation_AP": float(row["Mean_validation_AP"]),
        "SD_validation_AP": float(row["SD_validation_AP"]),
        "Mean_training_AP": float(row["Mean_training_AP"]),
    })

# Include the untuned dummy baseline.
dummy_row = initial_summary.loc[
    initial_summary["Model"] == "Dummy prior"
].iloc[0]

selection_rows.append({
    "Model": "Dummy prior",
    "Configuration": 0,
    "Parameters": json.dumps({}),
    "Mean_validation_AP": float(dummy_row["Mean_validation_AP"]),
    "SD_validation_AP": float(dummy_row["SD_validation_AP"]),
    "Mean_training_AP": float(dummy_row["Mean_training_AP"]),
})

selection_table = pd.DataFrame(selection_rows)

# Apply the prespecified model-family exact-tie rule.
selection_table["_tie_priority"] = (
    selection_table["Model"].map(tie_priority)
)

selection_table = (
    selection_table
    .sort_values(
        ["Mean_validation_AP", "_tie_priority"],
        ascending=[False, True],
    )
    .drop(columns="_tie_priority")
    .reset_index(drop=True)
)

selected_row = selection_table.iloc[0]
selected_model_name = selected_row["Model"]
selected_parameters = json.loads(selected_row["Parameters"])

# This is an UNFITTED definition of the selected configuration.
# It is not the final model fitted on all development records.
selected_pipeline_definition = clone(
    candidate_pipelines[selected_model_name]
).set_params(**selected_parameters)

selection_record = {
    "selected_model": selected_model_name,
    "configuration": int(selected_row["Configuration"]),
    "selected_parameters": selected_parameters,
    "complete_estimator_settings": (
        selected_pipeline_definition
        .named_steps["model"]
        .get_params(deep=False)
    ),
    "mean_validation_average_precision": float(
        selected_row["Mean_validation_AP"]
    ),
    "selection_rule": (
        "Highest unrounded mean development validation AP; "
        "prespecified exact-tie order"
    ),
    "protocol_sha256": file_sha256(PROTOCOL_PATH),
    "development_sha256": file_sha256(DEV_PATH),
    "preprocessing_module_sha256": file_sha256(MODULE_PATH),
    "cv_manifest_sha256": file_sha256(CV_MANIFEST_PATH),
    "status": (
        "Configuration selected; threshold selection, "
        "full-development refit and final test assessment pending"
    ),
}

SELECTION_PATH = RESULTS_DIR / "selected_model_configuration.json"

# Do not silently replace a different selection record.
if SELECTION_PATH.exists():
    existing_selection = json.loads(
        SELECTION_PATH.read_text(encoding="utf-8")
    )
    assert existing_selection == selection_record, (
        "A different selection record exists. Review before replacing it."
    )
else:
    SELECTION_PATH.write_text(
        json.dumps(selection_record, indent=2),
        encoding="utf-8",
    )

selection_table.to_csv(
    RESULTS_DIR / "model_selection_comparison.csv",
    index=False,
)

display(selection_table.round(4))

print("Selected model:", selected_model_name)
print("Selected parameters:", selected_parameters)
print("Selection record saved or verified.")
print("Selected pipeline definition remains unfitted.")

,Model,Configuration,Parameters,Mean_validation_AP,SD_validation_AP,Mean_training_AP
0,Random forest,1,"{""model__class_weight"": null, ""model__max_dept...",0.6662,0.0244,0.7568
1,Logistic regression,3,"{""model__C"": 1.0, ""model__class_weight"": null}",0.6607,0.0194,0.6675
2,Shallow decision tree,9,"{""model__class_weight"": null, ""model__max_dept...",0.6124,0.0213,0.6836
3,Dummy prior,0,{},0.2654,0.0001,0.2654


Selected model: Random forest
Selected parameters: {'model__class_weight': None, 'model__max_depth': 10, 'model__min_samples_leaf': 10}
Selection record saved or verified.
Selected pipeline definition remains unfitted.


**Development model selection**

The random forest with maximum depth 10, minimum leaf size 10 and no class weighting achieved the highest mean validation average precision among the compared configurations and baseline. Its mean validation AP was approximately 0.6662, compared with 0.6607 for logistic regression and 0.6124 for the best searched decision tree.

The random forest is selected according to the predefined rule using unrounded mean validation AP. Its advantage over logistic regression is small, approximately 0.0055 AP, and is not presented as proof of statistical significance or business value.

Restricting the forest reduced its training–validation AP gap compared with the initial configuration. Nevertheless, the selected forest remains more complex and less directly interpretable than logistic regression.

No best-per-family configuration used balanced class weighting within the specified searches. These findings apply to the configurations and development folds investigated.

Threshold selection, full-development refitting and reserved-test assessment remain pending. The selection results are development evidence rather than independent final performance estimates.

**Out of fold scoring and threshold selection**

For the selected random forest configuration, we generate out-of-fold development scores using the same five saved folds. Each customer is scored by a complete pipeline fitted on the other four folds.

We choose the threshold that maximises F1 across the pooled out-of-fold scores. If several thresholds achieve exactly the same maximum F1, we select the highest threshold. A customer is flagged when their score meets or exceeds the selected threshold.

These scores support threshold selection. Although each customer is excluded from the fit that produces their score, model configuration selection has already used development data. The resulting threshold and development metrics are therefore not independent final performance estimates.

In [14]:
#Generate the out-of-fold scores
# One score per development customer.
# Start with missing values so incomplete scoring is detectable.
oof_scores = np.full(len(X), np.nan)
oof_appearances = np.zeros(len(X), dtype=int)
oof_fold_rows = []

for fold_number, (train_positions, valid_positions) in enumerate(
    cv_splits,
    start=1,
):
    # Fresh complete pipeline for this fold.
    fold_pipeline = clone(selected_pipeline_definition)

    with warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        fold_pipeline.fit(
            X.iloc[train_positions],
            y.iloc[train_positions],
        )

    classes = fold_pipeline.named_steps["model"].classes_
    positive_columns = np.flatnonzero(classes == 1)
    assert len(positive_columns) == 1
    positive_column = int(positive_columns[0])

    # Score only the customers excluded from this fold's training.
    fold_scores = fold_pipeline.predict_proba(
        X.iloc[valid_positions]
    )[:, positive_column]

    assert np.isfinite(fold_scores).all()
    assert ((fold_scores >= 0) & (fold_scores <= 1)).all()

    oof_scores[valid_positions] = fold_scores
    oof_appearances[valid_positions] += 1

    fold_ap = average_precision_score(
        y.iloc[valid_positions],
        fold_scores,
    )

    oof_fold_rows.append({
        "Fold": fold_number,
        "Validation records": len(valid_positions),
        "Validation AP": fold_ap,
    })

    print(f"Fold {fold_number}: out-of-fold AP = {fold_ap:.4f}")

assert np.all(oof_appearances == 1)
assert np.isfinite(oof_scores).all()

oof_fold_results = pd.DataFrame(oof_fold_rows)

# Verify that rerunning the selected configuration reproduces
# its earlier tuning scores on the same folds.
selected_tuning_rows = (
    tuning_fold_results.loc[
        (tuning_fold_results["Model"] == selected_model_name)
        & (
            tuning_fold_results["Configuration"]
            == int(selected_row["Configuration"])
        )
    ]
    .sort_values("Fold")
)

assert len(selected_tuning_rows) == N_SPLITS

np.testing.assert_allclose(
    oof_fold_results.sort_values("Fold")["Validation AP"].to_numpy(),
    selected_tuning_rows["Validation AP"].to_numpy(),
    rtol=1e-10,
    atol=1e-12,
)

oof_predictions = pd.DataFrame({
    "customerID": customer_ids.to_numpy(),
    "validation_fold": validation_fold,
    "actual_churn": y.to_numpy(),
    "oof_churn_score": oof_scores,
})

oof_predictions.to_csv(
    RESULTS_DIR / "selected_model_oof_scores.csv",
    index=False,
)

oof_fold_results.to_csv(
    RESULTS_DIR / "selected_model_oof_fold_results.csv",
    index=False,
)

# No individual fold model is the final deployment/evaluation model.
del fold_pipeline

print("\nEvery development customer received exactly one OOF score.")
print("OOF fold results match the selected tuning configuration.")
print("Saved OOF scores and fold results.")

#Why fit five models again? The earlier tuning code saved performance summaries, but not each customer’s validation score. We now need those individual scores to investigate thresholds.

Fold 1: out-of-fold AP = 0.6581
Fold 2: out-of-fold AP = 0.6946
Fold 3: out-of-fold AP = 0.6879
Fold 4: out-of-fold AP = 0.6361
Fold 5: out-of-fold AP = 0.6544

Every development customer received exactly one OOF score.
OOF fold results match the selected tuning configuration.
Saved OOF scores and fold results.


In [15]:
#Choose the threshold

from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

# Calculate precision and recall at each distinct score threshold.
precision_values, recall_values, thresholds = precision_recall_curve(
    y,
    oof_scores,
)

# The final precision/recall point has no corresponding threshold.
# Remove that endpoint before calculating threshold-specific F1.
threshold_precision = precision_values[:-1]
threshold_recall = recall_values[:-1]

denominator = threshold_precision + threshold_recall

threshold_f1 = np.divide(
    2 * threshold_precision * threshold_recall,
    denominator,
    out=np.zeros_like(denominator),
    where=denominator > 0,
)

threshold_results = pd.DataFrame({
    "Threshold": thresholds,
    "Precision": threshold_precision,
    "Recall": threshold_recall,
    "F1": threshold_f1,
})

# Use unrounded values.
# For exact maximum-F1 ties, choose the highest threshold.
maximum_f1 = threshold_results["F1"].max()

best_threshold_row = (
    threshold_results.loc[
        threshold_results["F1"] == maximum_f1
    ]
    .sort_values("Threshold", ascending=False)
    .iloc[0]
)

selected_threshold = float(best_threshold_row["Threshold"])

# The agreed flagging rule includes scores equal to the threshold.
oof_flags = (oof_scores >= selected_threshold).astype(int)

oof_precision = precision_score(y, oof_flags, zero_division=0)
oof_recall = recall_score(y, oof_flags, zero_division=0)
oof_f1 = f1_score(y, oof_flags, zero_division=0)

# Verify that applying the threshold reproduces the selected F1.
np.testing.assert_allclose(oof_f1, maximum_f1, atol=1e-12)

tn, fp, fn, tp = confusion_matrix(
    y, oof_flags, labels=[0, 1]
).ravel()

threshold_summary = pd.DataFrame([{
    "Model": selected_model_name,
    "Threshold": selected_threshold,
    "OOF precision": oof_precision,
    "OOF recall": oof_recall,
    "OOF F1": oof_f1,
    "Flagged customers": int(oof_flags.sum()),
    "Flagged percentage": 100 * oof_flags.mean(),
    "True positives": int(tp),
    "False positives": int(fp),
    "False negatives": int(fn),
    "True negatives": int(tn),
}])

threshold_results.to_csv(
    RESULTS_DIR / "development_threshold_comparison.csv",
    index=False,
)

threshold_summary.to_csv(
    RESULTS_DIR / "selected_threshold_development_summary.csv",
    index=False,
)

# Add flags to the saved customer-level development evidence.
oof_predictions["high_risk_flag"] = oof_flags
oof_predictions.to_csv(
    RESULTS_DIR / "selected_model_oof_scores.csv",
    index=False,
)

print("Selected threshold:", repr(selected_threshold))
display(threshold_summary.round(4))

print(
    "Threshold selected using development OOF scores. "
    "These are selection results, not final test results."
)

#Do not round the threshold in the actual prediction rule. Rounding is fine for displaying results, but the saved full-precision value should be used later.
#Also, the chosen score cutoff is not a proven business-optimal rule or a guaranteed real-world churn probability.

Selected threshold: 0.319015704046037


,Model,Threshold,OOF precision,OOF recall,OOF F1,Flagged customers,Flagged percentage,True positives,False positives,False negatives,True negatives
0,Random forest,0.319,0.5547,0.7525,0.6387,2028,35.9957,1125,903,370,3236


Threshold selected using development OOF scores. These are selection results, not final test results.


**Selected threshold and development implications**

The selected random forest produced one out-of-fold score for each of the 5,634 development customers. Its fold-level average-precision results reproduced the selected tuning configuration.

The F1-maximising threshold was 0.319015704046037. Applying the rule score ≥ threshold produced development out-of-fold precision of approximately 0.5547, recall of 0.7525 and F1 of 0.6387.

The rule flagged 2,028 customers, approximately 36.00% of the development subset. These included 1,125 recorded churners and 903 recorded non-churners. It missed 370 recorded churners and correctly left 3,236 recorded non-churners unflagged.

This threshold is an analytical operating point selected using F1. It is not a verified cost-optimal retention rule. The flagged proportion describes this sample and is not an established operational contact capacity.

Model configuration and threshold selection both used development data. These results support selection and workload interpretation; independent reserved-test assessment remains pending.

In [16]:
# Freeze or verify the selected procedure.
# Preserve the original frozen record during a reproduction run.
# Only the reviewed Python 3.13.15 -> 3.13.16 change is permitted.
# All other fields must match exactly.

import json
import platform

MODELS_DIR = PROJECT_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

FREEZE_PATH = MODELS_DIR / "frozen_model_procedure.json"

# Confirm that the files underlying selection are unchanged.
saved_selection = json.loads(
    SELECTION_PATH.read_text(encoding="utf-8")
)

assert (
    saved_selection["protocol_sha256"] == file_sha256(PROTOCOL_PATH)
), "The modelling protocol differs from the saved selection."

assert (
    saved_selection["development_sha256"] == file_sha256(DEV_PATH)
), "The development file differs from the saved selection."

assert (
    saved_selection["preprocessing_module_sha256"]
    == file_sha256(MODULE_PATH)
), "The preprocessing module differs from the saved selection."

assert (
    saved_selection["cv_manifest_sha256"]
    == file_sha256(CV_MANIFEST_PATH)
), "The CV manifest differs from the saved selection."

# Confirm that the current model definition matches the saved selection.
assert (
    selected_model_name == saved_selection["selected_model"]
), "The selected model name differs."

assert (
    selected_pipeline_definition.named_steps["model"].get_params(
        deep=False
    )
    == saved_selection["complete_estimator_settings"]
), "The selected estimator settings differ."

# Keep this run's record separate from the original frozen record.
current_procedure = {
    "selected_model": selected_model_name,
    "complete_estimator_settings": (
        selected_pipeline_definition
        .named_steps["model"]
        .get_params(deep=False)
    ),
    "numerical_scaling": model_protocol["numerical_scaling"][
        selected_model_name
    ],
    "threshold": float(selected_threshold),
    "flag_rule": "score >= threshold",
    "positive_class": 1,
    "positive_class_meaning": "Churn = Yes",
    "predictor_columns": list(X.columns),
    "development_records": len(X),
    "development_sha256": file_sha256(DEV_PATH),
    "preprocessing_module_sha256": file_sha256(MODULE_PATH),
    "cv_manifest_sha256": file_sha256(CV_MANIFEST_PATH),
    "protocol_sha256": file_sha256(PROTOCOL_PATH),
    "selection_record_sha256": file_sha256(SELECTION_PATH),
    "oof_scores_sha256": file_sha256(
        RESULTS_DIR / "selected_model_oof_scores.csv"
    ),
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "scikit_learn_version": sklearn.__version__,
    "planned_final_metrics": [
        "average_precision",
        "precision_recall_curve",
        "precision",
        "recall",
        "f1",
        "confusion_matrix",
        "flagged_count",
        "flagged_proportion",
    ],
    "test_policy": (
        "Apply the fitted pipeline and frozen threshold without "
        "further fitting or tuning on the reserved test set."
    ),
}

# This project already has a frozen procedure.
# Stop if it is missing rather than recreating historical evidence.
assert FREEZE_PATH.is_file(), (
    "The original frozen procedure is missing. "
    "Check the project path and restore the original file."
)

existing_freeze = json.loads(
    FREEZE_PATH.read_text(encoding="utf-8")
)

# Print every difference before deciding whether it is permitted.
missing = object()
different_fields = []

for key in sorted(set(existing_freeze) | set(current_procedure)):
    saved_value = existing_freeze.get(key, missing)
    current_value = current_procedure.get(key, missing)

    if saved_value != current_value:
        different_fields.append(key)
        print(f"\nField: {key}")
        print(
            "Original:",
            "<missing>" if saved_value is missing else repr(saved_value),
        )
        print(
            "This run:",
            "<missing>" if current_value is missing else repr(current_value),
        )

# No changes other than the specifically reviewed Python version are allowed.
unexpected_fields = [
    key for key in different_fields if key != "python_version"
]

assert not unexpected_fields, (
    "Unexpected frozen-record differences: "
    + ", ".join(unexpected_fields)
    + ". Stop and investigate."
)

saved_python = existing_freeze["python_version"]
current_python = current_procedure["python_version"]

if saved_python != current_python:
    assert (saved_python, current_python) == (
        "3.13.15", "3.13.16"
    ), (
        f"Unreviewed Python version change: "
        f"{saved_python} -> {current_python}"
    )

# Later cells must use the original authoritative record.
# Nothing is written to frozen_model_procedure.json here.
frozen_procedure = existing_freeze

print("\nOriginal frozen procedure preserved.")
print("Original Python:", saved_python)
print("Current Python:", current_python)
print("All non-Python-version fields match exactly.")
print("Frozen threshold:", repr(frozen_procedure["threshold"]))
print("Prediction-equivalence checks must still pass in the later cell.")
print("This cell does not perform reserved-test evaluation.")


Field: python_version
Original: '3.13.15'
This run: '3.13.16'

Original frozen procedure preserved.
Original Python: 3.13.15
Current Python: 3.13.16
All non-Python-version fields match exactly.
Frozen threshold: 0.319015704046037
Prediction-equivalence checks must still pass in the later cell.
This cell does not perform reserved-test evaluation.


In [17]:
#Refit on all development records
#Until now, each fitted forest used only four development folds. We now train one complete pipeline on all 5,634 development customers, keeping the selected settings unchanged.
#Its preprocessing will learn medians, categories and other applicable parameters from all development records. That is appropriate now: selection is complete, and test records remain excluded.

# Fresh selected pipeline, with no previously fitted fold state.
final_pipeline = clone(selected_pipeline_definition)

with warnings.catch_warnings():
    warnings.simplefilter("error", ConvergenceWarning)
    final_pipeline.fit(X, y)

# Confirm the fitted classifier uses the frozen settings and classes.
assert (
    final_pipeline.named_steps["model"].get_params(deep=False)
    == frozen_procedure["complete_estimator_settings"]
)

np.testing.assert_array_equal(
    final_pipeline.named_steps["model"].classes_,
    np.array([0, 1]),
)

# Record the actual encoded features learned from all development data.
final_feature_names = (
    final_pipeline
    .named_steps["preprocessing"]
    .named_steps["columns"]
    .get_feature_names_out()
)

# Verify the existing feature list without overwriting original evidence.
feature_path = RESULTS_DIR / "final_pipeline_feature_names.csv"

assert feature_path.is_file(), (
    "Original feature list is missing. Check the project folder."
)

saved_feature_names = pd.read_csv(feature_path)["Feature"].tolist()

assert saved_feature_names == list(final_feature_names), (
    "The rerun produced a different feature list. Stop and investigate."
)

print("Original feature list verified and preserved.")

print("Selected pipeline fitted on all development records:", len(X))
print("Encoded feature count:", len(final_feature_names))
print("Frozen threshold remains:", repr(selected_threshold))
print("No reserved test records were used.")

Original feature list verified and preserved.
Selected pipeline fitted on all development records: 5634
Encoded feature count: 46
Frozen threshold remains: 0.319015704046037
No reserved test records were used.


**Saving and reload verification**

We save the complete fitted pipeline, including preprocessing and the selected random forest. The classification threshold is stored separately in the frozen-procedure record.

We verify that loading the saved pipeline reproduces the original pipeline’s scores and flags on a small development-data sample. This is a technical persistence check, not an additional performance evaluation.

The shared preprocessing module remains a required dependency because the pipeline references its custom parsing function. The reserved test data are not used during this check.

In [18]:
# Verify the existing saved pipeline against the development refit.
# Preserve original artefacts and save this rerun's evidence separately.

import json
import platform
import joblib
from datetime import datetime
from zoneinfo import ZoneInfo

MODEL_PATH = MODELS_DIR / "selected_churn_pipeline.joblib"
BUNDLE_RECORD_PATH = MODELS_DIR / "model_handover_record.json"

# This is a reproduction run: original artefacts must already exist.
assert MODEL_PATH.is_file(), (
    "Original saved model is missing. Check the project folder."
)
assert BUNDLE_RECORD_PATH.is_file(), (
    "Original handover record is missing. Check the project folder."
)

saved_freeze = json.loads(
    FREEZE_PATH.read_text(encoding="utf-8")
)
existing_record = json.loads(
    BUNDLE_RECORD_PATH.read_text(encoding="utf-8")
)

assert saved_freeze == frozen_procedure, (
    "The in-memory procedure differs from the original frozen record."
)

assert (
    file_sha256(MODULE_PATH)
    == frozen_procedure["preprocessing_module_sha256"]
), "The shared preprocessing module changed."

assert (
    file_sha256(DEV_PATH)
    == frozen_procedure["development_sha256"]
), "The development file changed."

assert (
    existing_record["frozen_procedure_sha256"]
    == file_sha256(FREEZE_PATH)
), "The frozen procedure does not match the original handover."

assert (
    existing_record["model_sha256"] == file_sha256(MODEL_PATH)
), "The saved model does not match its original fingerprint."

# Record original fingerprints so preservation can be checked afterward.
original_hashes = {
    "model": file_sha256(MODEL_PATH),
    "frozen_procedure": file_sha256(FREEZE_PATH),
    "handover_record": file_sha256(BUNDLE_RECORD_PATH),
}

# Use the same development-only verification sample:
# the first 20 records plus records with missing TotalCharges.
assert X.index.is_unique, "Development predictor index is not unique."
assert customer_ids.index.equals(X.index), (
    "Customer IDs and predictors are not aligned."
)

missing_total = telco_prep.parse_predictors(X)["TotalCharges"].isna()

check_indices = list(dict.fromkeys(
    list(X.index[:20]) + list(X.index[missing_total])
))
X_reload_check = X.loc[check_indices].copy()

# Load only the original trusted project model.
reloaded_pipeline = joblib.load(MODEL_PATH)

# Check both classifiers against the original frozen settings.
for pipeline in (final_pipeline, reloaded_pipeline):
    assert (
        pipeline.named_steps["model"].get_params(deep=False)
        == frozen_procedure["complete_estimator_settings"]
    ), "Estimator settings differ from the frozen procedure."

    np.testing.assert_array_equal(
        pipeline.named_steps["model"].classes_,
        np.array([0, 1]),
    )

# Locate the positive-class column separately for each pipeline.
refit_positive_column = int(np.flatnonzero(
    final_pipeline.named_steps["model"].classes_ == 1
)[0])

saved_positive_column = int(np.flatnonzero(
    reloaded_pipeline.named_steps["model"].classes_ == 1
)[0])

scores_refit = final_pipeline.predict_proba(
    X_reload_check
)[:, refit_positive_column]

scores_saved = reloaded_pipeline.predict_proba(
    X_reload_check
)[:, saved_positive_column]

assert np.isfinite(scores_refit).all()
assert np.isfinite(scores_saved).all()

# Keep the original strict numerical tolerance.
np.testing.assert_allclose(
    scores_refit,
    scores_saved,
    rtol=1e-12,
    atol=1e-12,
)

threshold = frozen_procedure["threshold"]

flags_refit = (scores_refit >= threshold).astype(int)
flags_saved = (scores_saved >= threshold).astype(int)

np.testing.assert_array_equal(flags_refit, flags_saved)

# Confirm that the original artefacts remain unchanged.
assert original_hashes["model"] == file_sha256(MODEL_PATH)
assert (
    original_hashes["frozen_procedure"]
    == file_sha256(FREEZE_PATH)
)
assert (
    original_hashes["handover_record"]
    == file_sha256(BUNDLE_RECORD_PATH)
)

# Save new evidence in a separate, uniquely named rerun folder.
verified_at = datetime.now(ZoneInfo("Asia/Colombo"))
stamp = verified_at.strftime("%Y%m%dT%H%M%S%f")

rerun_dir = RESULTS_DIR / "rerun_verification" / stamp
rerun_dir.mkdir(parents=True, exist_ok=False)

comparison_path = rerun_dir / "prediction_comparison.csv"

pd.DataFrame({
    "customerID": customer_ids.loc[check_indices].to_numpy(),
    "refitted_pipeline_score": scores_refit,
    "original_saved_pipeline_score": scores_saved,
    "refitted_pipeline_flag": flags_refit,
    "original_saved_pipeline_flag": flags_saved,
}).to_csv(comparison_path, index=False)

verification_record = {
    "record_type": "development_refit_verification",
    "verified_at": verified_at.isoformat(),
    "original_model_sha256": original_hashes["model"],
    "original_frozen_procedure_sha256": (
        original_hashes["frozen_procedure"]
    ),
    "original_handover_record_sha256": (
        original_hashes["handover_record"]
    ),
    "development_sha256": file_sha256(DEV_PATH),
    "preprocessing_module_sha256": file_sha256(MODULE_PATH),
    "original_python_version": saved_freeze["python_version"],
    "current_python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "scikit_learn_version": sklearn.__version__,
    "joblib_version": joblib.__version__,
    "frozen_threshold": threshold,
    "verification_records": len(X_reload_check),
    "score_rtol": 1e-12,
    "score_atol": 1e-12,
    "maximum_absolute_score_difference": float(
        np.max(np.abs(scores_refit - scores_saved))
    ),
    "scores_match_within_tolerance": True,
    "flags_match_exactly": True,
    "original_artefacts_preserved": True,
    "comparison_file": comparison_path.name,
    "comparison_sha256": file_sha256(comparison_path),
    "reserved_test_used_in_this_check": False,
    "scope": (
        "Agreement on the specified development verification sample; "
        "not a new test evaluation or proof of agreement on all inputs."
    ),
}

record_path = rerun_dir / "rerun_verification_record.json"
record_path.write_text(
    json.dumps(verification_record, indent=2),
    encoding="utf-8",
)

print("Verification records:", len(X_reload_check))
print("Scores match within numerical tolerance.")
print("High-risk flags match exactly.")
print("Original model, frozen procedure and handover record preserved.")
print("Saved separate rerun evidence:", rerun_dir)
print("No reserved test records were used in this verification.")

Verification records: 28
Scores match within numerical tolerance.
High-risk flags match exactly.
Original model, frozen procedure and handover record preserved.
Saved separate rerun evidence: /content/drive/MyDrive/ML 2026-DS-77 Telco Customer Churn/outputs/tables/rerun_verification/20261014T114246117889
No reserved test records were used in this verification.


**Member 4 handover — model development and selection**

**Responsible member:** IT24102789  
**Receiving member:** IT24103677 — Group Leader  
**Preprocessing support:** IT24103676

**Completed work**

We verified the development data and shared preprocessing module against Member 3’s recorded file fingerprints. We defined a dummy baseline, logistic regression, a decision tree and random forest, each within a complete preprocessing-and-model pipeline.

We created and saved five stratified development cross-validation folds using shuffling and seed 42. Every candidate used the same fold membership. Learned preprocessing was fitted within each training fold.

The initial comparison completed 20 fits. The predefined tuning search completed 130 fits across 26 configurations. Model selection used unrounded mean validation average precision and the recorded tie rules.

**Selected model**

Random forest achieved the highest mean validation average precision among the evaluated configurations:

- 200 trees.
- Maximum depth: 10.
- Minimum samples per leaf: 10.
- Maximum features considered per split: square root.
- No class weighting.
- Random seed: 42.
- Unscaled numerical preprocessing, with median imputation and categorical one-hot encoding.

Mean development validation AP was approximately 0.6662, compared with 0.6607 for logistic regression, 0.6124 for the best searched decision tree and 0.2654 for the dummy baseline.

The forest’s advantage over logistic regression was small. It is not presented as proof of statistical significance, commercial value or superiority over methods that were not evaluated.

**Threshold selection**

Using the selected configuration and the same five folds, we generated one out-of-fold score for every development customer. Fold-level AP reproduced the selected tuning results.

The F1-maximising threshold was:

0.319015704046037

A customer is flagged when their score meets or exceeds this threshold. The full-precision value must be used.

Development out-of-fold results were:

- Precision: approximately 0.5547.
- Recall: approximately 0.7525.
- F1: approximately 0.6387.
- Flagged customers: 2,028 of 5,634, approximately 36.00%.
- True positives: 1,125.
- False positives: 903.
- False negatives: 370.
- True negatives: 3,236.

These results were used during selection and are not independent final test estimates. The threshold is an analytical F1-based choice, not a verified cost-optimal retention rule.

**Final fitting and persistence**

The selected complete pipeline was fitted on all 5,634 development records and produced 46 encoded features. It was saved and reloaded successfully.

A persistence check on 28 development records, including missing-TotalCharges examples, reproduced scores within numerical tolerance and flags exactly. This verifies persistence behaviour on the checked records, not predictive generalisation.

**Files required by the group leader**

- `models/selected_churn_pipeline.joblib`
- `models/frozen_model_procedure.json`
- `models/model_handover_record.json`
- `src/telco_preprocessing.py`
- The executed modelling notebook.
- The saved development CV manifest, modelling protocol, comparison tables, OOF scores and threshold results.
- Relevant decision-log entries and the preprocessing handover.

The custom preprocessing module must remain importable when loading the fitted pipeline.

**Instructions for final evaluation**

The group leader should verify the handover files and environment, then load the saved pipeline and full-precision frozen threshold.

Apply the pipeline to the reserved test predictors without fitting or tuning. Exclude customerID and Churn from predictors and preserve customer IDs separately for traceability.

Report the agreed test metrics and flagged counts, then describe flagged and unflagged customers. Do not change the selected procedure in response to test results.

**Remaining work**

The fresh-session top-to-bottom run was completed successfully on 12/10/2026, as recorded below. Group-leader handover acceptance should be recorded when performed.

Reserved-test evaluation, final high-risk profiling, the joint recommendation and final submission preparation remain pending.